# 01 — Visual Exploration of Global CPI

Initial visual inspection of the global monthly inflation rate.

**Source:** World Bank hcpi_m — cross-country median YoY rate across 186 countries (replicates HCPI_GLOBAL_MED).

**Note:** `cpi_global_rate` is already a year-on-year rate (YoY %), computed from national indices
and aggregated as a cross-sectional median. It is not a level index.

**Input:** `data/processed/cpi_global_monthly.parquet`  
**Establishes:** Series range, regime changes, train/val/test splits

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path('..').resolve()         # tfg-forecasting/
MONOREPO = ROOT.parent              # tfg-ipc-mcp/
sys.path.insert(0, str(MONOREPO))

from shared.constants import DATE_TRAIN_END, DATE_VAL_END, FREQ

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

## 1. Data Loading

In [ ]:
DATA_PATH = ROOT / "data" / "processed" / "cpi_global_monthly.parquet"
df = pd.read_parquet(DATA_PATH)
print(f"Range: {df.index.min().date()} -> {df.index.max().date()}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [3]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
cpi_global_rate,276.0,3.441554,1.859766,0.799304,2.223008,2.933433,3.991994,9.639495


In [ ]:
# Verify frequency and gaps
expected = pd.date_range(df.index.min(), df.index.max(), freq="MS")
missing = expected.difference(df.index)
print(f"Expected months: {len(expected)}")
print(f"Present months: {len(df)}")
print(f"Gaps: {len(missing)}")
if len(missing) > 0:
    print("Missing dates:", missing.tolist())

## 2. Full Series — Global Inflation Rate (YoY %)

In [ ]:
y = df["cpi_global_rate"]

fig, ax = plt.subplots()

# Shading periods
SHADING = [
    ("Financial crisis", "2008-09-01", "2009-06-30", "#fff3cd", 0.55),
    ("Covid-19",         "2020-01-01", "2020-12-31", "#e8e8e8", 0.50),
    ("Inflation shock",  "2021-01-01", "2022-12-31", "#f8d7d7", 0.55),
    ("Normalisation",    "2023-01-01", "2024-12-31", "#d7e8f8", 0.40),
]
for label, s, e, color, alpha in SHADING:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=color, alpha=alpha,
               zorder=0, label=label)

ax.plot(y.index, y.values, linewidth=1.5, color="#2166ac", zorder=3, label="Global CPI YoY (%)")
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(2, color="grey", linewidth=0.7, linestyle=":", label="ECB Target (2%)")

# Train/val/test split lines
for label, date, color in [
    ("Train end", DATE_TRAIN_END, "green"),
    ("Val end",   DATE_VAL_END,   "orange"),
]:
    ax.axvline(pd.Timestamp(date), color=color, linestyle="--", alpha=0.8, label=label)

ax.set_title("Global CPI — Year-on-year inflation rate (YoY median, %)")
ax.set_ylabel("YoY Rate (%)")
ax.legend(fontsize=9, ncol=3)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 3. First Difference of the Rate (month-on-month change)

In [ ]:
y_diff1 = y.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y.index, y.values, linewidth=1.2, color="#2166ac")
axes[0].set_title("YoY Rate in level (%)")
axes[0].axhline(0, color="black", linewidth=0.5)

axes[1].plot(y_diff1.index, y_diff1.values, linewidth=1.0, color="#d62728")
axes[1].set_title("First difference of the rate (delta YoY)")
axes[1].axhline(0, color="black", linewidth=0.5)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

## 4. Monthly Distribution (box plot)

In [ ]:
monthly_df = pd.DataFrame({
    "month": y.index.month,
    "rate": y.values,
})

fig, ax = plt.subplots(figsize=(12, 5))
monthly_df.boxplot(column="rate", by="month", ax=ax)
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
ax.set_xticklabels(months)
ax.set_title("Distribution of cpi_global_rate by calendar month")
ax.set_xlabel("Month")
ax.set_ylabel("YoY Rate (%)")
ax.axhline(y.mean(), color="red", linestyle="--", alpha=0.5, label=f"Global mean: {y.mean():.2f}%")
ax.legend()
plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Train / Validation / Test Splits

In [ ]:
train = df.loc[:DATE_TRAIN_END]
val   = df.loc[DATE_TRAIN_END:DATE_VAL_END].iloc[1:]
test  = df.loc[DATE_VAL_END:].iloc[1:]

fig, ax = plt.subplots()
ax.plot(train.index, train["cpi_global_rate"], label=f"Train ({len(train)})", color="#2ca02c", linewidth=1.5)
ax.plot(val.index,   val["cpi_global_rate"],   label=f"Val ({len(val)})",     color="#ff7f0e", linewidth=1.5)
ax.plot(test.index,  test["cpi_global_rate"],  label=f"Test ({len(test)})",   color="#d62728", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Global CPI — Temporal Splits")
ax.set_ylabel("YoY Rate (%)")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

print(f"Train: {train.index.min().date()} -> {train.index.max().date()} ({len(train)} months)")
print(f"Val:   {val.index.min().date()} -> {val.index.max().date()} ({len(val)} months)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()} ({len(test)} months)")

## 6. Summary

In [ ]:
print("="*60)
print("GLOBAL CPI — EXPLORATION SUMMARY")
print("="*60)
print(f"Temporal range:  {df.index.min().date()} -> {df.index.max().date()}")
print(f"Total months:    {len(df)}")
print(f"Frequency:       Monthly (MS)")
print(f"Gaps:            {len(missing)}")
print(f"NaN:             {df['cpi_global_rate'].isna().sum()}")
print(f"Mean:            {y.mean():.4f} %")
print(f"Median:          {y.median():.4f} %")
print(f"Std:             {y.std():.4f} %")
print(f"Min:             {y.min():.4f} %  ({y.idxmin().date()})")
print(f"Max:             {y.max():.4f} %  ({y.idxmax().date()})")
print("="*60)
print()
print("Methodological note:")
print("  cpi_global_rate = median YoY rates across 186 countries (unweighted by GDP)")
print("  Differs ~0.26 pp from World Bank HCPI_GLOBAL_MED annual mean.")
print("  See 01_etl/01_ingest_cpi_global.py for details.")